False Acceptance Rate baseline

In [ ]:
import os
import cv2
import csv
from deepface import DeepFace


# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================

# Location of the ORL database.
DATABASE = "attdb"

# Number of subjects in the ORL database.
NUM_SUBJECTS = 40

# Recognition model used for every comparison.
MODEL_NAME = "Facenet"

# Store every individual impostor comparison here.
trial_results = []


# ============================================================
# 2. LOAD EACH SUBJECT'S ENROLLMENT IMAGE
# ============================================================

# For FAR testing we only need one enrollment image per subject,
# since we are comparing enrolled identities against OTHER
# subjects, not against modified versions of themselves.

enrollment_images = {}

for i in range(1, NUM_SUBJECTS + 1):

    folder = os.path.join(DATABASE, f"s{i}")

    images = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".pgm", ".png", ".jpg", ".jpeg"))
    ]

    images.sort(
        key=lambda name: int(os.path.splitext(name)[0])
    )

    # Use the same first image as the FRR script for consistency.
    enrollment_path = os.path.join(folder, images[0])

    img = cv2.imread(enrollment_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    enrollment_images[f"s{i}"] = img


# ============================================================
# 3. RUN THE ZERO-EFFORT IMPOSTOR EXPERIMENT
# ============================================================

# For every enrolled subject, compare their enrollment image
# against every OTHER subject's enrollment image. Each of these
# comparisons is a "zero-effort impostor attack" -- a genuinely
# different person attempting authentication with their own face.
#
# A FAR failure happens when DeepFace incorrectly marks one of
# these cross-identity comparisons as "verified".

subject_ids = list(enrollment_images.keys())

for enrolled_id in subject_ids:

    enrolled_img = enrollment_images[enrolled_id]

    print(f"\nTesting impostors against enrolled identity {enrolled_id}")

    for impostor_id in subject_ids:

        # Skip comparing a subject against themselves --
        # that would be a genuine match, not an impostor attempt.
        if impostor_id == enrolled_id:
            continue

        impostor_img = enrollment_images[impostor_id]

        try:

            result = DeepFace.verify(
                img1_path=enrolled_img,
                img2_path=impostor_img,
                model_name=MODEL_NAME,
                detector_backend="skip"
            )

            verified = result["verified"]
            distance = result["distance"]
            threshold = result["threshold"]

            trial_results.append({
                "enrolled_subject": enrolled_id,
                "impostor_subject": impostor_id,
                "verified": verified,
                "distance": distance,
                "threshold": threshold,
                "error": ""
            })

        except Exception as e:

            trial_results.append({
                "enrolled_subject": enrolled_id,
                "impostor_subject": impostor_id,
                "verified": None,
                "distance": None,
                "threshold": None,
                "error": str(e)
            })

    print(f"Finished {enrolled_id}")


# ============================================================
# 4. CALCULATE THE FALSE ACCEPTANCE RATE
# ============================================================

# Only successfully completed comparisons are included.
completed = [
    trial for trial in trial_results
    if trial["verified"] is not None
]

# A "false acceptance" is any impostor comparison that DeepFace
# incorrectly marked as verified.
false_accepts = sum(
    trial["verified"] for trial in completed
)

total = len(completed)
errors = len(trial_results) - total

far = (false_accepts / total * 100) if total else 0

print("\n========== EXPERIMENT RESULTS ==========")
print(f"Total impostor comparisons: {total}")
print(f"False accepts: {false_accepts}")
print(f"Errors: {errors}")
print(f"False Acceptance Rate: {far:.2f}%")


# ============================================================
# 5. EXPORT RESULTS TO CSV
# ============================================================

with open("trial_results_far_baseline.csv", "w", newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=trial_results[0].keys()
    )

    writer.writeheader()
    writer.writerows(trial_results)

print("\nResults saved to trial_results_far_baseline.csv")